# Emotion Recognition Using Audio Features

In [ ]:
from huggingface_hub import snapshot_download
from IPython.display import Audio
import soundfile as sf
import matplotlib.pyplot as plt
import numpy as np
import os, csv
import pandas as pd
import torch
import torchaudio
import torchaudio.functional as AF
from torch.utils.data import Dataset, DataLoader

from sklearn.datasets import make_circles, make_classification, make_moons
from sklearn.ensemble import AdaBoostClassifier, RandomForestClassifier

from sklearn.metrics import accuracy_score
from sklearn.metrics import balanced_accuracy_score
from sklearn.metrics import classification_report
from sklearn.metrics import f1_score

from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier

from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier

In [ ]:
EMO2IDX = {"angry": 0, "happy": 1, "neutral": 2, "sad": 3}
IDX2EMO = {"0": "angry", "1": "happy", "2": "neutral", "3": "sad"}

ROOT = snapshot_download("OlhaHavryliuk/UA-SER", repo_type="dataset")


class UASER(Dataset):
    def __init__(self, split):
        self.dir = os.path.join(ROOT, "clips")
        self.rows = [r for r in csv.DictReader(open(os.path.join(ROOT, "dataset.csv")))
                     if r["split"] == split]

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, i):
        r = self.rows[i]
        wav, _ = sf.read(os.path.join(self.dir, r["filename"]), dtype="float32")
        return torch.from_numpy(wav), EMO2IDX[r["emotion"]]


def collate(batch):
    wavs, labels = zip(*batch)
    T = max(w.shape[0] for w in wavs)
    audio = torch.zeros(len(wavs), T)
    for i, w in enumerate(wavs):
        audio[i, : w.shape[0]] = w
    return audio, torch.tensor(labels)


train_loader = DataLoader(UASER("train"), batch_size=16, shuffle=True, collate_fn=collate)
test_loader  = DataLoader(UASER("test"),  batch_size=16, collate_fn=collate)


if __name__ == "__main__":
    print("train:", len(train_loader.dataset), "| test:", len(test_loader.dataset))
    audio, labels = next(iter(train_loader))
    print(audio.shape, labels.tolist())

In [ ]:
audio1, label1 = audio[0], labels[0]
sample_rate = 16000

In [ ]:
def get_mfcc(audio, sample_rate=sample_rate):
    if audio.ndim == 1:
        waveform = audio.unsqueeze(0)
        
    mfcc_transform = torchaudio.transforms.MFCC(
        sample_rate=sample_rate,
        n_mfcc=13,
        melkwargs={
            "n_fft": 400,
            "hop_length": 160,
            "n_mels": 40,
        }
    )
        
    mfcc = mfcc_transform(waveform)
    return mfcc


def get_f0(audio, sample_rate=sample_rate):
    if audio.ndim == 1:
        waveform = audio.unsqueeze(0)
    
    f0 = AF.detect_pitch_frequency(
        waveform,
        sample_rate=sample_rate,
        frame_time=0.01, 
        freq_low=50,
        freq_high=500,
    )
    
    time = torch.arange(f0.shape[-1]) * 0.01 
    return f0, time


def get_rms(audio, sample_rate=sample_rate):
    if audio.ndim == 1:
        waveform = audio.unsqueeze(0)

    frames = waveform.unfold(
        dimension=-1,
        size=400,
        step=160
    )

    rms = torch.sqrt(torch.mean(frames ** 2, dim=-1) + 1e-8)
    time = torch.arange(rms.shape[-1]) * 160 / sample_rate
    return rms, time


def get_0crossing(audio, sample_rate=sample_rate):
    if audio.ndim == 1:
        waveform = audio.unsqueeze(0)

    frames = waveform.unfold(
        dimension=-1,
        size=400,
        step=160
    )

    zero_crossings = (frames[..., 1:] * frames[..., :-1]) < 0
    zcr = zero_crossings.float().mean(dim=-1)
    time = torch.arange(zcr.shape[-1]) * 160 / sample_rate
    return zcr, time

In [ ]:
def plot_acoustic_features(audio=audio1, label=label1):
    emo = IDX2EMO[str(int(label))]
    fig, axes = plt.subplots(5, 1, figsize=(18, 27)) 
    
    fig.suptitle(emo, fontsize=20, y=0.96)
    fig.subplots_adjust(top=0.92)    
    
    axes[0].plot(audio, color='Black')
    axes[0].set(ylabel='Count', title="Audio")

    mfcc = get_mfcc(audio)
    axes[1].imshow(mfcc[0].detach().numpy(), aspect="auto", origin="lower")
    axes[1].set(ylabel='Count', title="MFCC")

    f0, time = get_f0(audio)
    axes[2].plot(time.numpy(), f0[0].numpy(), color='red')
    axes[2].set(ylabel='Frequency, Hz', title="Pitch/F0")

    rms, time = get_rms(audio)
    axes[3].plot(time.numpy(), rms[0].numpy(), color='green')
    axes[3].set(ylabel='Count', title="RMS energy")

    zcr, time = get_0crossing(audio)
    axes[4].plot(time.numpy(), zcr[0].numpy(), color='blue')
    axes[4].set(xlabel='Time', ylabel='Count', title="Zero-Crossing Rate")

plot_acoustic_features()

In [ ]:
play_audio = audio1.detach().cpu().numpy()
Audio(play_audio, rate=sample_rate)

## Models

In [ ]:
def metrics(y_true, y_pred):
    accuracy = accuracy_score(y_true, y_pred)
    uar = balanced_accuracy_score(y_true, y_pred)
    macro_f1 = f1_score(y_true, y_pred, average="macro", zero_division=0)
    return [accuracy, uar, macro_f1]

In [ ]:
def get_mfcc_features(audio):
    if audio.ndim == 1:
        audio = audio.unsqueeze(0)

    mfcc_transform = torchaudio.transforms.MFCC(
    sample_rate=sample_rate,
    n_mfcc=13,
    melkwargs={
        "n_fft": 400,
        "hop_length": 160,
        "n_mels": 40,
        }
    )
    mfcc = mfcc_transform(audio.float())

    mfcc_mean = mfcc.mean(dim=-1).squeeze(0)
    mfcc_std = mfcc.std(dim=-1).squeeze(0)
    return torch.cat([mfcc_mean, mfcc_std]).numpy()

In [ ]:
def to_np(loader):
    all_x, all_y = [], []

    for x, y in loader:
        x, y = x.detach().cpu(), y.detach().cpu()

        for i in range(x.shape[0]):
            features = get_mfcc_features(x[i])
            all_x.append(features)

        if y.ndim > 1:
            y = torch.argmax(y, dim=1)

        all_y.append(y.numpy())

    all_x = np.vstack(all_x)
    all_y = np.concatenate(all_y, axis=0)
    return all_x, all_y

target_len = sample_rate * 3  # 3 seconds

x_train, y_train = to_np(train_loader)
x_test, y_test = to_np(test_loader)

In [ ]:
names = [
    "Nearest Neighbors",
    "Random Forest",
    "AdaBoost",
]

classifiers = [
    KNeighborsClassifier(n_neighbors=4),
    RandomForestClassifier(
        max_depth=5, max_features="sqrt", class_weight="balanced", n_estimators=100, random_state=42),
    AdaBoostClassifier(random_state=42),
]

result = {}
for name, clf in zip(names, classifiers):
    result[name] = {}

    clf.fit(x_train, y_train)
    preds = clf.predict(x_test)
    score = metrics(y_test, preds)
        
    result[name] = {
            "accuracy": score[0],
            "uar": score[1],
            "macro_f1": score[2],
        }
result

In [ ]:
df_result = pd.DataFrame(result).T
df_result.to_csv("base_metrics.csv", index=False)
df_result